# Drony fire detector retraining

Trains two new fire/smoke detectors on D-Fire and keeps the better one only if it beats the current `best.pt`.

**Before you start**, put these in Google Drive under `MyDrive/drony/`:
- `D-Fire-drony.zip` (made on the laptop by `prepare_dfire.py`)
- `best.pt` (from `backend/ML/`)
- the `training/` folder (from `backend/ML/`)

Then choose **Runtime → Change runtime type → T4 GPU** and **Runtime → Run all**.
If Colab disconnects, just **Run all** again: finished work is skipped and unfinished training resumes.

## 1. Check for a GPU

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime → Change runtime type → T4 GPU, then Run all again")
print("GPU:", torch.cuda.get_device_name(0))

## 2. Connect Google Drive and check the inputs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
DRIVE = '/content/drive/MyDrive/drony'
sys.path.insert(0, f'{DRIVE}/training')
try:
    from colab_steps import missing_inputs, run_state, mark_finished
except ImportError:
    raise SystemExit(f"Upload the backend/ML/training folder to {DRIVE}/training first")

missing = missing_inputs(DRIVE)
if missing:
    raise SystemExit("Missing in MyDrive/drony: " + ", ".join(missing))
print("All inputs found")

## 3. Install the same Ultralytics version as the laptop

In [ ]:
!pip -q install ultralytics==8.4.176

## 4. Unpack the dataset onto Colab's fast local disk

In [ ]:
import shutil, zipfile, time

shutil.copy(f'{DRIVE}/D-Fire-drony.zip', '/content/D-Fire-drony.zip')
shutil.rmtree('/content/dfire', ignore_errors=True)
with zipfile.ZipFile('/content/D-Fire-drony.zip') as z:
    z.extractall('/content/dfire')

DATA = '/content/dfire/data.yaml'
lines = open(DATA).read().splitlines()
lines[0] = 'path: /content/dfire'
open(DATA, 'w').write('\n'.join(lines) + '\n')

for split in ('train', 'val', 'test'):
    print(split, len(os.listdir(f'/content/dfire/images/{split}')), 'images')

## 5. Trial run (about 2 minutes) to catch problems before the long training

In [ ]:
from ultralytics import YOLO

TRIAL = '/content/trial'
shutil.rmtree(TRIAL, ignore_errors=True)
for split, count in (('train', 50), ('val', 10)):
    os.makedirs(f'{TRIAL}/images/{split}')
    os.makedirs(f'{TRIAL}/labels/{split}')
    for name in sorted(os.listdir(f'/content/dfire/images/{split}'))[:count]:
        shutil.copy(f'/content/dfire/images/{split}/{name}', f'{TRIAL}/images/{split}/')
        shutil.copy(f'/content/dfire/labels/{split}/{name[:-4]}.txt', f'{TRIAL}/labels/{split}/')
open(f'{TRIAL}/data.yaml', 'w').write(
    f"path: {TRIAL}\ntrain: images/train\nval: images/val\nnames:\n  0: fire\n  1: smoke\n")

start = time.time()
YOLO('yolo11n.pt').train(data=f'{TRIAL}/data.yaml', epochs=1, imgsz=640, batch=16,
                         project='/content/trial_runs', exist_ok=True)
elapsed = time.time() - start
print(f"Trial OK in {elapsed:.0f} s. Rough upper estimate for one full epoch: ~{elapsed * 13776 / 50 / 60:.0f} min")

## 6. Candidate A: fine-tune the current model (up to 30 rounds)

In [ ]:
def train_candidate(name, weights, epochs):
    run_dir = f'{DRIVE}/runs/{name}'
    state = run_state(run_dir, epochs)
    if state == 'done':
        print(f'{name}: already finished, skipping')
        return
    if state == 'resume':
        print(f'{name}: resuming from the last saved round')
        YOLO(f'{run_dir}/weights/last.pt').train(resume=True)
    else:
        print(f'{name}: starting')
        YOLO(weights).train(data=DATA, epochs=epochs, patience=10, imgsz=640, batch=32,
                            project=f'{DRIVE}/runs', name=name, exist_ok=True)
    mark_finished(run_dir)

train_candidate('A', f'{DRIVE}/best.pt', epochs=30)

## 7. Candidate B: train a fresh YOLO11n (up to 50 rounds)

In [ ]:
train_candidate('B', 'yolo11n.pt', epochs=50)

## 8. Compare on the 4,306 test pictures and keep the winner

In [ ]:
from compare import main as compare_main

winner = compare_main(
    {'old': f'{DRIVE}/best.pt',
     'A': f'{DRIVE}/runs/A/weights/best.pt',
     'B': f'{DRIVE}/runs/B/weights/best.pt'},
    DATA, f'{DRIVE}/results.csv')

if winner:
    name, imgsz = winner
    shutil.copy(f'{DRIVE}/runs/{name}/weights/best.pt', f'{DRIVE}/best_v2.pt')
    print(f"Saved {name} as MyDrive/drony/best_v2.pt (best at size {imgsz}).")
    if imgsz == 960:
        print("On the laptop, also set DETECT_IMGSZ=960 in backend/.env.")
else:
    print("Keep the old model. Nothing copied.")

## 9. Optional: severity classifier

Off by default. Only useful once sorted crops exist in `MyDrive/drony/severity/` as
`train/{mild,moderate,severe}/` and `val/{mild,moderate,severe}/` folders.

In [ ]:
RUN_SEVERITY = False

if RUN_SEVERITY:
    YOLO('yolov8n-cls.pt').train(data=f'{DRIVE}/severity', epochs=30, imgsz=224,
                                 project=f'{DRIVE}/runs', name='severity', exist_ok=True)
    shutil.copy(f'{DRIVE}/runs/severity/weights/best.pt', f'{DRIVE}/best_severity_v2.pt')
    print("Saved MyDrive/drony/best_severity_v2.pt")
else:
    print("Severity training skipped (RUN_SEVERITY = False)")